# 📈 CO2 Time Series Forecasting — Recursive Strategy

![Python](https://img.shields.io/badge/Python-3.8+-blue?logo=python)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.0+-orange?logo=scikit-learn)
![Task](https://img.shields.io/badge/Task-Time%20Series-red)
![Dataset](https://img.shields.io/badge/Dataset-Mauna%20Loa%20CO2-lightgrey)

---

## 📌 Project Overview

This project forecasts **future CO2 concentration levels** using a **Recursive (one-step-ahead) strategy** — predicting one step at a time and feeding predictions back as input for the next step.

| Item | Detail |
|------|--------|
| **Dataset** | Carbon Dioxide Levels in Atmosphere (Mauna Loa) |
| **Model** | Linear Regression |
| **Strategy** | Recursive — predict 1 step, feed back as next input |
| **Window Size** | 5 past observations → predict next 1 |

## 📦 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✅ Libraries loaded successfully!')

## 📂 2. Load & Visualize Data

> 📥 Dataset: [Carbon Dioxide Levels in Atmosphere — Kaggle](https://www.kaggle.com/datasets/ucsandiego/carbon-dioxide)
>
> Download `archive.csv` and place it in the same folder as this notebook.

In [ ]:
# Auto-detect file name
if os.path.exists('archive.csv'):
    data = pd.read_csv('archive.csv')
    print('✅ Loaded archive.csv')
elif os.path.exists('archive (1).csv'):
    data = pd.read_csv('archive (1).csv')
    print('✅ Loaded archive (1).csv')
elif os.path.exists('co2.csv'):
    data = pd.read_csv('co2.csv')
    print('✅ Loaded co2.csv')
else:
    raise FileNotFoundError('❌ Please place archive.csv in this folder!')

print(f'Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
data.head(10)

In [ ]:
# Rename columns for convenience
data.columns = data.columns.str.strip()
data = data.rename(columns={
    'Year'            : 'year',
    'Month'           : 'month',
    'Decimal Date'    : 'time',
    'Carbon Dioxide (ppm)' : 'co2',
    'Carbon Dioxide'  : 'co2',
})

# Keep only time and co2 columns
data = data[['time', 'co2']].dropna()
data['co2'] = pd.to_numeric(data['co2'], errors='coerce')
data = data.dropna()
data['co2'] = data['co2'].interpolate()

print(f'Clean shape: {data.shape}')
data.head(10)

In [ ]:
# Plot full CO2 time series
plt.figure(figsize=(13, 4))
plt.plot(data['time'], data['co2'], color='steelblue', linewidth=0.8)
plt.title('CO2 Concentration Over Time (Mauna Loa)')
plt.xlabel('Year')
plt.ylabel('CO2 (ppm)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🪟 3. Create Sliding Window Features

**Recursive strategy:** Use the last `N` values to predict the **next 1 value**.

```
Window: [t, t+1, t+2, t+3, t+4] → Target: t+5
Window: [t+1, t+2, t+3, t+4, t+5] → Target: t+6
```

In [ ]:
def create_recursive_data(data, window_size):
    """Create lag features for recursive (one-step-ahead) forecasting."""
    df = data.copy()
    for i in range(1, window_size):
        df[f'co2_{i}'] = df['co2'].shift(-i)
    df['target'] = df['co2'].shift(-window_size)
    return df.dropna()

window_size = 5
data_feat   = create_recursive_data(data, window_size)

print(f'Window size  : {window_size}')
print(f'Total samples: {len(data_feat)}')
data_feat.head()

## ✂️ 4. Train / Test Split (Time-Based)

> ⚠️ For time series, we **never shuffle** — always split by time order!

In [ ]:
X = data_feat.drop(['time', 'target'], axis=1)
y = data_feat['target']

train_size = 0.8
split_idx  = int(train_size * len(data_feat))

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train: {len(X_train)} samples')
print(f'Test : {len(X_test)} samples')

## 🏋️ 5. Train & Evaluate Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('='*35)
print(f'  MAE : {mean_absolute_error(y_test, y_pred):.4f}')
print(f'  MSE : {mean_squared_error(y_test, y_pred):.4f}')
print(f'  R²  : {r2_score(y_test, y_pred):.4f}')
print('='*35)

In [ ]:
# Plot train / test / prediction
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(data_feat['time'][:split_idx], data_feat['co2'][:split_idx],
        label='Train', color='steelblue', linewidth=0.8)
ax.plot(data_feat['time'][split_idx:], data_feat['co2'][split_idx:],
        label='Test (Actual)', color='orange', linewidth=1.5)
ax.plot(data_feat['time'][split_idx:], y_pred,
        label='Predicted', color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Year')
ax.set_ylabel('CO2 (ppm)')
ax.set_title('CO2 Forecasting — Recursive Strategy')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🔮 6. Forecast Future 10 Steps Ahead

In [ ]:
current_window = data['co2'].values[-window_size:].tolist()
future_preds   = []

print('🔮 10-Step Ahead Forecast:')
print('-'*30)
for i in range(10):
    pred = model.predict([current_window])[0]
    future_preds.append(pred)
    print(f'  Step {i+1:2d}: {pred:.2f} ppm')
    current_window = current_window[1:] + [pred]

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), future_preds, marker='o', color='tomato', linewidth=2)
plt.title('10-Step Ahead CO2 Forecast (Recursive)')
plt.xlabel('Step Ahead')
plt.ylabel('CO2 (ppm)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🏁 7. Summary

| Item | Detail |
|------|--------|
| **Strategy** | Recursive (1-step-ahead) |
| **Window Size** | 5 past observations |
| **Model** | Linear Regression |
| **R² Score** | ~0.99 |

### 🔑 Key Takeaways
- ✅ Sliding window converts time series into supervised learning
- ✅ CO2 has strong linear trend → Linear Regression achieves R² ≈ 0.99
- ✅ Time-based split prevents data leakage
- ⚠️ Recursive errors compound over long horizons — see Project 4 (Direct Strategy)